# <font color="#418FDE" size="6.5" uppercase>**Generalized Bases**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit Poisson, Gamma, and Tweedie regressors for appropriate target distributions. 
- Explain link functions and deviance through practical model evaluation. 
- Build polynomial and spline regression pipelines for nonlinear patterns. 


## **1. Generalized Regression Models**

### **1.1. Poisson Count Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_01.jpg?v=1783961186" width="250">



>* Model nonnegative event counts
>* Predict expected frequency from features

>* Model event rates using predictors and exposure
>* Compare risks fairly for practical decisions

>* Check assumptions and possible overdispersion
>* Interpret effects as multiplicative count changes



In [ ]:
#@title Python Code - Poisson Count Models

# Demonstrate Poisson regression for count outcomes.
# Counts stay nonnegative through exponential means.
# Deviance compares fitted counts against observations.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random generator.
rng = np.random.default_rng(42)

# Create small exposure and traffic features.
n = 80
miles = rng.uniform(1.0, 10.0, n)
traffic = rng.uniform(0.0, 1.0, n)

# Build a true positive Poisson mean.
linear_true = -0.2 + 0.18 * miles + 0.9 * traffic
mu_true = np.exp(linear_true)
counts = rng.poisson(mu_true)

# Validate the simulated count data.
assert counts.shape == miles.shape == traffic.shape
assert np.all(counts >= 0)

# Add intercept and two predictors.
X = np.column_stack([np.ones(n), miles, traffic])
y = counts.astype(float)

# Fit Poisson regression using gradient descent.
beta = np.zeros(X.shape[1])
learning_rate = 0.002
for step in range(2500):
    mu = np.exp(np.clip(X @ beta, -20, 20))
    gradient = X.T @ (mu - y) / n
    beta = beta - learning_rate * gradient

# Compute fitted means and Poisson deviance.
mu_hat = np.exp(np.clip(X @ beta, -20, 20))
safe_y = np.where(y == 0, 1.0, y)
terms = np.where(y == 0, mu_hat, y * np.log(safe_y / mu_hat) - y + mu_hat)
dev = 2.0 * np.sum(terms)

# Print a compact teaching summary.
print("Poisson count model fitted without scikit-learn.")
print(f"Observed mean count: {y.mean():.2f}")
print(f"Predicted mean count: {mu_hat.mean():.2f}")
print(f"Poisson deviance: {dev:.2f}")
print(f"Traffic coefficient: {beta[2]:.2f}")

# Plot observed counts against fitted means.
plt.figure(figsize=(6, 4))
plt.scatter(mu_hat, y, alpha=0.75)
plt.xlabel("Fitted expected count")
plt.ylabel("Observed count")
plt.title("Poisson model: positive fitted counts")
plt.grid(True, alpha=0.3)
plt.show()



### **1.2. Gamma Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_02.jpg?v=1783961190" width="250">



>* Models positive, skewed continuous outcomes
>* Handles variability increasing with expected value

>* Model positive outcomes with log links
>* Interpret predictor effects as proportional changes

>* Use Gamma only for positive outcomes
>* Evaluate skewness, scale, and model fit



### **1.3. Tweedie Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_03.jpg?v=1783961188" width="250">



>* Models zero-heavy, positive continuous outcomes
>* Better than linear, Poisson, or Gamma

>* Mean and variance change together
>* Power controls zero-inflated event amounts

>* Check for nonnegative, zero-heavy skewed targets
>* Evaluate both zero frequency and positive scale



## **2. GLM Evaluation**

### **2.1. Link Function Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_01.jpg?v=1783961183" width="250">



>* Links connect linear models to outcome types
>* They keep predictions within valid ranges

>* Links shape coefficient interpretation
>* Log links express multiplicative outcome changes

>* Link functions shape predictions before evaluation
>* Check range, meaning, and error patterns



### **2.2. Deviance Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_02.jpg?v=1783961192" width="250">



>* Deviance compares fitted and ideal GLM models
>* Lower deviance often means better fit

>* Deviance matches each model’s distribution assumptions
>* Use it with practical judgment

>* Compare deviance across models and validation data
>* Balance lower deviance with real-world reliability



In [ ]:
#@title Python Code - Deviance Metrics

# Deviance compares fitted and ideal GLM predictions.
# This example uses tiny synthetic count data.
# Lower deviance means better distribution-aware fit.

import math
import numpy as np
import matplotlib.pyplot as plt

# Create deterministic feature values in miles.
rng = np.random.default_rng(7)
miles = np.linspace(1.0, 10.0, 30)
rate = np.exp(0.4 + 0.18 * miles)

# Simulate Poisson counts from expected rates.
claims = rng.poisson(rate)
assert miles.shape == claims.shape
assert claims.size == 30

# Build a simple design matrix manually.
ones = np.ones_like(miles)
X = np.column_stack([ones, miles])
beta = np.zeros(2)

# Fit Poisson regression using Newton updates.
for step in range(8):
    mu = np.exp(X @ beta)
    gradient = X.T @ (claims - mu)
    hessian = X.T @ (X * mu[:, None])

    beta += np.linalg.solve(hessian, gradient)

# Compute fitted and null model predictions.
fitted_mu = np.exp(X @ beta)
null_mu = np.full_like(claims, claims.mean(), dtype=float)

# Define Poisson deviance with safe zero handling.
def poisson_deviance(y, mu):
    y = np.asarray(y, dtype=float)
    mu = np.maximum(np.asarray(mu, dtype=float), 1e-12)

    term = np.where(y == 0, 0.0, y * np.log(y / mu))
    return float(2.0 * np.sum(term - (y - mu)))

# Compare fitted model against intercept-only baseline.
fit_dev = poisson_deviance(claims, fitted_mu)
null_dev = poisson_deviance(claims, null_mu)
improvement = 100.0 * (null_dev - fit_dev) / null_dev

# Print concise teaching results.
print(f"Poisson GLM coefficients: intercept={beta[0]:.3f}, slope={beta[1]:.3f}")
print(f"Null deviance: {null_dev:.2f}")
print(f"Fitted deviance: {fit_dev:.2f}")
print(f"Deviance reduction: {improvement:.1f}%")
print("Log link keeps predicted claim counts positive.")

# Plot observed counts and fitted mean curve.
plt.figure(figsize=(7, 4))
plt.scatter(miles, claims, label="observed counts")
plt.plot(miles, fitted_mu, color="crimson", label="fitted mean")
plt.xlabel("Annual driving distance, thousand miles")
plt.ylabel("Insurance claim count")
plt.title("Poisson deviance evaluates count predictions")
plt.legend()
plt.tight_layout()
plt.show()



### **2.3. Assessing Model Fit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_03.jpg?v=1783961194" width="250">



>* Match evaluation to distribution and link
>* Check predictions, residuals, and group reliability

>* Compare deviance to judge model improvement
>* Balance better fit with generalization

>* Check residuals for patterns and misspecification
>* Validate predictions using deviance and held-out data



## **3. Basis Design**

### **3.1. Polynomial Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_01.jpg?v=1783961196" width="250">



>* Add powers to model curved patterns
>* Keep linear regression tools and evaluation

>* Higher degrees risk overfitting and edge instability
>* Use scaling and regularization for stability

>* Use pipelines for consistent polynomial transformations
>* Validate degree for stable, interpretable curves



In [ ]:
#@title Python Code - Polynomial Features

# Polynomial features add controlled curve flexibility.
# We compare straight and curved fits.
# Small synthetic data keeps learning transparent.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible one dimensional training data.
rng = np.random.default_rng(7)
x = np.linspace(0, 10, 40)
noise = rng.normal(0, 3, x.size)

# Build a curved target with noise.
y = 4 + 1.5 * x - 0.35 * x**2 + noise
assert x.shape == y.shape

# Create polynomial design matrices manually.
X_linear = np.column_stack([np.ones_like(x), x])
X_poly = np.column_stack([np.ones_like(x), x, x**2])

# Fit least squares coefficients safely.
coef_linear = np.linalg.lstsq(X_linear, y, rcond=None)[0]
coef_poly = np.linalg.lstsq(X_poly, y, rcond=None)[0]

# Predict on training points for evaluation.
pred_linear = X_linear @ coef_linear
pred_poly = X_poly @ coef_poly

# Compute mean squared errors for comparison.
mse_linear = np.mean((y - pred_linear) ** 2)
mse_poly = np.mean((y - pred_poly) ** 2)

# Print concise teaching results only.
print(f"Linear MSE: {mse_linear:.2f}")
print(f"Polynomial degree 2 MSE: {mse_poly:.2f}")
print("Polynomial features keep linear coefficients.")

# Prepare smooth curves for visualization.
x_grid = np.linspace(0, 10, 200)
G_linear = np.column_stack([np.ones_like(x_grid), x_grid])
G_poly = np.column_stack([np.ones_like(x_grid), x_grid, x_grid**2])

# Plot data and both fitted relationships.
plt.figure(figsize=(7, 4))
plt.scatter(x, y, label="Observed data", alpha=0.75)
plt.plot(x_grid, G_linear @ coef_linear, label="Linear fit")
plt.plot(x_grid, G_poly @ coef_poly, label="Polynomial fit")

# Label the plot for interpretation.
plt.xlabel("Temperature in degrees Celsius")
plt.ylabel("Electricity demand units")
plt.title("Polynomial Features Capture Curvature")
plt.legend()

# Display exactly one plot.
plt.tight_layout()
plt.show()



### **3.2. Spline Basis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_02.jpg?v=1783961198" width="250">



>* Splines model local bends using knots
>* Linear pipelines capture nonlinear predictor patterns

>* Choose knots to balance flexibility and smoothness
>* Splines reveal data-shaped nonlinear patterns

>* Fit spline pipelines consistently across data splits
>* Validate smooth, interpretable curves against simpler models



### **3.3. Interaction Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_03.jpg?v=1783961200" width="250">



>* Interactions let predictors influence each other
>* They improve nonlinear polynomial and spline models

>* Combine raw, transformed, or categorical features
>* Use plausible interactions with enough data

>* Add interactions carefully to avoid overfitting
>* Validate and interpret realistic feature combinations



In [ ]:
#@title Python Code - Interaction Features

# Interaction features reveal combined predictor effects.
# This example uses simple linear algebra.
# We compare additive and interaction bases.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic temperature and humidity data.
rng = np.random.default_rng(7)
n_samples = 80

# Simulate weather predictors in metric units.
temp_c = rng.uniform(10, 35, n_samples)
humidity = rng.uniform(20, 90, n_samples)

# Build a target with a true interaction.
noise = rng.normal(0, 8, n_samples)
energy = 40 + 1.2 * temp_c + 0.3 * humidity
energy = energy + 0.08 * temp_c * humidity + noise

# Build additive and interaction design matrices.
ones = np.ones_like(temp_c)
X_add = np.column_stack([ones, temp_c, humidity])
X_int = np.column_stack([ones, temp_c, humidity, temp_c * humidity])

# Validate matching row counts before fitting.
assert X_add.shape[0] == energy.size
assert X_int.shape[0] == energy.size

# Fit least squares models without sklearn.
coef_add = np.linalg.lstsq(X_add, energy, rcond=None)[0]
coef_int = np.linalg.lstsq(X_int, energy, rcond=None)[0]

# Predict training values for comparison.
pred_add = X_add @ coef_add
pred_int = X_int @ coef_int

# Compute mean squared errors manually.
mse_add = np.mean((energy - pred_add) ** 2)
mse_int = np.mean((energy - pred_int) ** 2)

# Print a compact teaching summary.
print(f"Additive basis MSE: {mse_add:.2f}")
print(f"Interaction basis MSE: {mse_int:.2f}")
print(f"Estimated interaction coefficient: {coef_int[3]:.3f}")

# Plot predictions against observed outcomes.
plt.figure(figsize=(6, 4))
plt.scatter(energy, pred_add, label="Additive", alpha=0.75)
plt.scatter(energy, pred_int, label="With interaction", alpha=0.75)

# Add a reference line for perfect predictions.
low = min(energy.min(), pred_add.min(), pred_int.min())
high = max(energy.max(), pred_add.max(), pred_int.max())
plt.plot([low, high], [low, high], "k--", linewidth=1)

# Label the plot for interpretation.
plt.xlabel("Observed energy use")
plt.ylabel("Predicted energy use")
plt.title("Interaction feature improves nonlinear fit")
plt.legend()

# Display the single teaching plot.
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Generalized Bases**</font>


In this lecture, you learned to:
- Fit Poisson, Gamma, and Tweedie regressors for appropriate target distributions. 
- Explain link functions and deviance through practical model evaluation. 
- Build polynomial and spline regression pipelines for nonlinear patterns. 

In the next Module (Module 11), we will go over 'Linear Classification'